In [1]:
from pathlib import Path
import pandas as pd

# 项目根目录
PROJECT_ROOT = Path.cwd().resolve()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data" / "raw"

# 检查 CSV 文件
csv_files = sorted(DATA_DIR.glob("*.csv"))

print(f"发现 {len(csv_files)} 个 CSV 文件")

for file in csv_files:
    print(file.name)

发现 9 个 CSV 文件
olist_customers_dataset.csv
olist_geolocation_dataset.csv
olist_order_items_dataset.csv
olist_order_payments_dataset.csv
olist_order_reviews_dataset.csv
olist_orders_dataset.csv
olist_products_dataset.csv
olist_sellers_dataset.csv
product_category_name_translation.csv


In [2]:
orders = pd.read_csv(
    DATA_DIR / "olist_orders_dataset.csv"
)

print("数据维度：", orders.shape)

print("\n前五行：")
display(orders.head())

print("\n字段信息：")
orders.info()

数据维度： (99441, 8)

前五行：


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00



字段信息：
<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype
---  ------                         --------------  -----
 0   order_id                       99441 non-null  str  
 1   customer_id                    99441 non-null  str  
 2   order_status                   99441 non-null  str  
 3   order_purchase_timestamp       99441 non-null  str  
 4   order_approved_at              99281 non-null  str  
 5   order_delivered_carrier_date   97658 non-null  str  
 6   order_delivered_customer_date  96476 non-null  str  
 7   order_estimated_delivery_date  99441 non-null  str  
dtypes: str(8)
memory usage: 6.1 MB


In [3]:
# 各字段缺失值
print("缺失值统计：")
print(orders.isnull().sum())

# 完全重复的记录
print("\n重复行数量：")
print(orders.duplicated().sum())

# 订单 ID 是否唯一
print("\n重复订单 ID：")
print(orders["order_id"].duplicated().sum())

# 订单状态分布
print("\n订单状态分布：")
print(orders["order_status"].value_counts())

缺失值统计：
order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64

重复行数量：
0

重复订单 ID：
0

订单状态分布：
order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64


In [5]:

# 数据集文件映射
datasets = {
    "customers": "olist_customers_dataset.csv",
    "orders": "olist_orders_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "order_payments": "olist_order_payments_dataset.csv",
    "order_reviews": "olist_order_reviews_dataset.csv",
    "products": "olist_products_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "geolocation": "olist_geolocation_dataset.csv",
    "category_translation": "product_category_name_translation.csv"
}

results = []

for table_name, filename in datasets.items():

    df = pd.read_csv(DATA_DIR / filename)

    # 统计缺失字段
    missing = df.isnull().sum()
    missing = missing[missing > 0]

    results.append({
        "表名": table_name,
        "行数": df.shape[0],
        "列数": df.shape[1],
        "重复行数": df.duplicated().sum(),
        "缺失字段数": len(missing),
        "缺失单元格数": missing.sum()
    })

# 汇总结果
profile_summary = pd.DataFrame(results)

display(profile_summary)

,表名,行数,列数,重复行数,缺失字段数,缺失单元格数
0,customers,99441,5,0,0,0
1,orders,99441,8,0,3,4908
2,order_items,112650,7,0,0,0
3,order_payments,103886,5,0,0,0
4,order_reviews,99224,7,0,2,145903
5,products,32951,9,0,8,2448
6,sellers,3095,4,0,0,0
7,geolocation,1000163,5,261831,0,0
8,category_translation,71,2,0,0,0


In [6]:
# 订单状态与实际送达时间缺失情况
delivery_check = pd.crosstab(
    orders["order_status"],
    orders["order_delivered_customer_date"].isnull(),
    rownames=["订单状态"],
    colnames=["实际送达时间是否缺失"]
)

display(delivery_check)

实际送达时间是否缺失,False,True
订单状态,,
approved,0,2
canceled,6,619
created,0,5
delivered,96470,8
invoiced,0,314
processing,0,301
shipped,0,1107
unavailable,0,609


In [7]:
# 读取全部数据
dfs = {
    name: pd.read_csv(DATA_DIR / filename)
    for name, filename in datasets.items()
}

# 各表候选主键
primary_keys = {
    "customers": ["customer_id"],
    "orders": ["order_id"],
    "order_items": ["order_id", "order_item_id"],
    "order_payments": ["order_id", "payment_sequential"],
    "order_reviews": ["review_id"],
    "products": ["product_id"],
    "sellers": ["seller_id"],
    "category_translation": ["product_category_name"]
}

pk_results = []

for table, keys in primary_keys.items():

    df = dfs[table]

    duplicate_count = df.duplicated(subset=keys).sum()

    null_count = df[keys].isna().any(axis=1).sum()

    pk_results.append({
        "表名": table,
        "候选主键": ", ".join(keys),
        "主键重复数": duplicate_count,
        "主键缺失行数": null_count,
        "是否满足唯一性": duplicate_count == 0 and null_count == 0
    })

display(pd.DataFrame(pk_results))

,表名,候选主键,主键重复数,主键缺失行数,是否满足唯一性
0,customers,customer_id,0,0,True
1,orders,order_id,0,0,True
2,order_items,"order_id, order_item_id",0,0,True
3,order_payments,"order_id, payment_sequential",0,0,True
4,order_reviews,review_id,814,0,False
5,products,product_id,0,0,True
6,sellers,seller_id,0,0,True
7,category_translation,product_category_name,0,0,True


In [9]:
# 定义需要检查的关联关系
relationships = [
    ("orders", "customer_id", "customers", "customer_id"),
    ("order_items", "order_id", "orders", "order_id"),
    ("order_payments", "order_id", "orders", "order_id"),
    ("order_reviews", "order_id", "orders", "order_id"),
    ("order_items", "product_id", "products", "product_id"),
    ("order_items", "seller_id", "sellers", "seller_id"),
    (
        "products",
        "product_category_name",
        "category_translation",
        "product_category_name"
    )
]

fk_results = []

for child, child_key, parent, parent_key in relationships:

    child_values = dfs[child][child_key]
    parent_values = dfs[parent][parent_key]

    # 非空关联字段中，无法匹配父表的记录
    unmatched = (
        child_values.notna()
        & ~child_values.isin(parent_values.dropna())
    )

    fk_results.append({
        "子表": child,
        "关联字段": child_key,
        "父表": parent,
        "关联字段缺失": int(child_values.isna().sum()),
        "无法匹配记录数": int(unmatched.sum()),
        "匹配率": round(
            (1 - unmatched.sum() / child_values.notna().sum()) * 100,
            2
        ) if child_values.notna().sum() else None
    })

display(pd.DataFrame(fk_results))

,子表,关联字段,父表,关联字段缺失,无法匹配记录数,匹配率
0,orders,customer_id,customers,0,0,100.00
1,order_items,order_id,orders,0,0,100.00
2,order_payments,order_id,orders,0,0,100.00
3,order_reviews,order_id,orders,0,0,100.00
4,order_items,product_id,products,0,0,100.00
5,order_items,seller_id,sellers,0,0,100.00
6,products,product_category_name,category_translation,610,13,99.96


In [10]:
reviews = dfs["order_reviews"]

print("review_id 重复数：",
      reviews.duplicated("review_id").sum())

print("review_id + order_id 重复数：",
      reviews.duplicated(
          ["review_id", "order_id"]
      ).sum())

# 同一个 review_id 是否对应多个订单
review_order_count = (
    reviews.groupby("review_id")["order_id"]
    .nunique()
)

print("关联多个订单的 review_id 数量：",
      (review_order_count > 1).sum())

# 一个订单是否存在多条评价
order_review_count = (
    reviews.groupby("order_id")
    .size()
)

print("存在多条评价的订单数量：",
      (order_review_count > 1).sum())

# 查看重复 review_id 的具体记录
duplicate_reviews = reviews[
    reviews.duplicated("review_id", keep=False)
]

display(
    duplicate_reviews
    .sort_values(["review_id", "order_id"])
    .head(10)
)

review_id 重复数： 814
review_id + order_id 重复数： 0
关联多个订单的 review_id 数量： 789
存在多条评价的订单数量： 547


,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
29841,00130cbe1f9d422698c812ed8ded1919,04a28263e085d399c97ae49e0b477efa,1,NaN,"O cartucho ""original HP"" 60XL não é reconhecid...",2018-03-07 00:00:00,2018-03-20 18:08:23
46678,00130cbe1f9d422698c812ed8ded1919,dfcdfc43867d1c1381bfaf62d6b9c195,1,NaN,"O cartucho ""original HP"" 60XL não é reconhecid...",2018-03-07 00:00:00,2018-03-20 18:08:23
63193,0115633a9c298b6a98bcbe4eee75345f,0c9850b2c179c1ef60d2855e2751d1fa,5,NaN,NaN,2017-09-21 00:00:00,2017-09-26 03:27:47
90677,0115633a9c298b6a98bcbe4eee75345f,78a4201f58af3463bdab842eea4bc801,5,NaN,NaN,2017-09-21 00:00:00,2017-09-26 03:27:47
57280,0174caf0ee5964646040cd94e15ac95e,74db91e33b4e1fd865356c89a61abf1f,1,NaN,Produto entregue dentro de embalagem do fornec...,2018-03-07 00:00:00,2018-03-08 03:00:53
92876,0174caf0ee5964646040cd94e15ac95e,f93a732712407c02dce5dd5088d0f47b,1,NaN,Produto entregue dentro de embalagem do fornec...,2018-03-07 00:00:00,2018-03-08 03:00:53
54832,017808d29fd1f942d97e50184dfb4c13,8daaa9e99d60fbba579cc1c3e3bfae01,5,NaN,NaN,2018-03-02 00:00:00,2018-03-05 01:43:30
99167,017808d29fd1f942d97e50184dfb4c13,b1461c8882153b5fe68307c46a506e39,5,NaN,NaN,2018-03-02 00:00:00,2018-03-05 01:43:30
96080,0254bd905dc677a6078990aad3331a36,331b367bdd766f3d1cf518777317b5d9,1,NaN,O pedido consta de 2 produtos e até agora rece...,2017-09-09 00:00:00,2017-09-13 09:52:44
20621,0254bd905dc677a6078990aad3331a36,5bf226cf882c5bf4247f89a97c86f273,1,NaN,O pedido consta de 2 produtos e até agora rece...,2017-09-09 00:00:00,2017-09-13 09:52:44


In [1]:
detail_tables = [
    "order_items",
    "order_payments",
    "order_reviews"
]

grain_results = []

for table in detail_tables:

    df = dfs[table]

    # 每个订单对应多少条记录
    counts = df.groupby("order_id").size()

    grain_results.append({
        "表名": table,
        "总记录数": len(df),
        "关联订单数": counts.size,
        "平均每单记录数": round(counts.mean(), 2),
        "最大每单记录数": counts.max(),
        "存在多条记录的订单数": (counts > 1).sum()
    })

display(pd.DataFrame(grain_results))

NameError: name 'dfs' is not defined